# End-to-End Explainable CKD Risk Prediction & Severity Assessment

## Complete ML Pipeline — Jupyter Notebook

This notebook runs the entire CKD risk prediction pipeline:
1. Data Loading (2159 patients)
2. Preprocessing (70/20/10 Stratified Split)
3. Model Training (5 models)
4. Evaluation (Validation + Test)
5. Risk Scoring
6. SHAP Explainability
7. eGFR & CKD Staging
8. Reporting & Visualization

**Note:** eGFR and serum creatinine are excluded from prediction features to avoid data leakage.

---

In [ ]:
# Setup: Add project root to path
import sys, os
import warnings
import numpy as np

warnings.filterwarnings('ignore')
np.random.seed(42)

# If running from notebooks/ folder, add parent to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

print(f'Working directory: {os.getcwd()}')

---
## Module 1: Data Loading

In [ ]:
from src.data_loader import load_data, get_dataset_summary, get_display_name

df = load_data('data/final_verified_ckd_dataset_fixed.xlsx')
summary = get_dataset_summary(df)

print(f"Dataset shape: {df.shape}")
print(f"Total patients: {summary['total_patients']}")
print(f"CKD positive: {summary['ckd_positive']} ({summary['ckd_percentage']}%)")
print(f"CKD negative: {summary['ckd_negative']}")
print(f"Null values: {summary['null_count']}")

df.head()

In [ ]:
df.info()
print('\nDescriptive statistics:')
df.describe()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
df['ckd_diagnosis'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('CKD Diagnosis Distribution')
ax.set_xlabel('Diagnosis (0=No CKD, 1=CKD)')
ax.set_ylabel('Count')
ax.set_xticklabels(['Not CKD', 'CKD'], rotation=0)
plt.tight_layout()
plt.show()

---
## Module 2: Preprocessing (70/20/10 Stratified Split)

In [ ]:
from src.preprocessing import preprocess, encode_features, EXCLUDE_FROM_PREDICTION

df_original = df.copy()

X_train, X_val, X_test, y_train, y_val, y_test, scaler, feature_names = preprocess(df)

print(f'Training samples (after SMOTE): {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Test samples: {len(X_test)}')
print(f'Prediction features ({len(feature_names)}): {feature_names}')
print(f'Excluded from prediction: {EXCLUDE_FROM_PREDICTION}')
print(f'\nTrain class distribution:\n{y_train.value_counts()}')
print(f'\nValidation class distribution:\n{y_val.value_counts()}')
print(f'\nTest class distribution:\n{y_test.value_counts()}')

In [ ]:
from sklearn.model_selection import train_test_split

df_encoded = encode_features(df_original.copy())
all_indices = np.arange(len(df_encoded))

_, test_split_idx, _, _ = train_test_split(
    all_indices, df_encoded['ckd_diagnosis'],
    test_size=0.10, random_state=42, stratify=df_encoded['ckd_diagnosis']
)
remaining_idx = np.setdiff1d(all_indices, test_split_idx)
remaining_y = df_encoded['ckd_diagnosis'].iloc[remaining_idx]

_, val_split_idx, _, _ = train_test_split(
    remaining_idx, remaining_y,
    test_size=0.2222, random_state=42, stratify=remaining_y
)
val_indices = np.array(val_split_idx)
print(f'Validation indices (first 10): {val_indices[:10]}')

---
## Module 3: Model Training

In [ ]:
from src.model_training import train_all_models

models = train_all_models(X_train, y_train, feature_names)
print(f'\nTrained {len(models)} models: {list(models.keys())}')

---
## Module 4: Evaluation

In [ ]:
from src.evaluation import evaluate_models

best_name, best_model, val_comparison_df, test_results_df = evaluate_models(
    models, X_val, y_val, X_test, y_test, X_train, y_train
)

print(f'\nBest Model (selected on validation): {best_name}')
print(f'\nValidation Results:')
display(val_comparison_df)
print(f'\nTest Results (unbiased):')
display(test_results_df)

In [ ]:
from IPython.display import Image, display

print('Model Comparison Bar Chart (Validation):')
display(Image(filename='outputs/model_comparison_bar.png'))

print('\nROC Curves (Validation):')
display(Image(filename='outputs/roc_curves_validation.png'))

print('\nConfusion Matrices (Validation):')
display(Image(filename='outputs/confusion_matrices_validation.png'))

print('\nConfusion Matrices (Test):')
display(Image(filename='outputs/confusion_matrices_test.png'))

---
## Module 5: Risk Scoring

In [ ]:
from src.risk_scoring import assign_risk_scores

risk_df = assign_risk_scores(best_model, X_val, y_val)

print('Risk distribution (validation set):')
print(risk_df['Risk_Category'].value_counts())
print()
risk_df.head(10)

In [ ]:
display(Image(filename='outputs/risk_distribution_pie.png'))
display(Image(filename='outputs/risk_distribution_bar.png'))
display(Image(filename='outputs/risk_score_histogram.png'))

---
## Module 6: SHAP Explainability

In [ ]:
from src.explainability import compute_shap_values, explain_patient

# Use best base model for SHAP if Stacking is selected
shap_model = best_model
shap_model_name = best_name
if best_name == 'Stacking':
    base_df = val_comparison_df[val_comparison_df['Model'] != 'Stacking']
    base_sorted = base_df.sort_values(by=['Recall', 'F1-Score'], ascending=False)
    shap_model_name = base_sorted.iloc[0]['Model']
    shap_model = models[shap_model_name]
    print(f'Using {shap_model_name} for SHAP (TreeExplainer does not support Stacking)')

shap_values, explainer = compute_shap_values(shap_model, X_val, feature_names)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
patient_idx = 0
explanations = explain_patient(shap_values, feature_names, patient_idx)

print(f'Top explanations for patient {patient_idx}:')
for exp in explanations:
    print(f'  - {exp["description"]}')

In [ ]:
display(Image(filename='outputs/shap_summary_beeswarm.png'))
display(Image(filename='outputs/shap_bar_plot.png'))
display(Image(filename='outputs/shap_waterfall_patient_0.png'))

---
## Module 7: eGFR & CKD Staging

In [ ]:
from src.staging import compute_staging

staging_df = compute_staging(df_original)

print('CKD Stage Distribution:')
print(staging_df['CKD_Stage'].value_counts())
staging_df.head(10)

In [ ]:
display(Image(filename='outputs/ckd_stage_distribution.png'))
display(Image(filename='outputs/egfr_distribution_histogram.png'))

---
## Module 8: Reporting & Visualization

In [ ]:
from src.reporting import (
    generate_patient_report, plot_correlation_heatmap,
    plot_feature_importance, plot_dashboard_summary,
)

df_enc_for_corr = df_encoded.drop(columns=EXCLUDE_FROM_PREDICTION, errors='ignore')
plot_correlation_heatmap(df_enc_for_corr)
display(Image(filename='outputs/correlation_heatmap.png'))

In [ ]:
fi_model = shap_model if not hasattr(best_model, 'feature_importances_') else best_model
plot_feature_importance(fi_model, feature_names)
display(Image(filename='outputs/feature_importance.png'))

In [ ]:
plot_dashboard_summary(val_comparison_df, risk_df, staging_df)
display(Image(filename='outputs/dashboard_summary.png'))

In [ ]:
report = generate_patient_report(
    patient_idx=0, df_original=df_original, X_test=X_val, y_test=y_val,
    model=best_model, shap_values=shap_values, feature_names=feature_names,
    staging_df=staging_df, test_indices=val_indices,
)

with open('outputs/patient_report_0.txt') as f:
    print(f.read())

---
## Pipeline Summary

The complete CKD risk prediction pipeline has been executed:

1. **Data loaded**: 2159 patients, 7 prediction features
2. **Preprocessing**: Encoding, scaling, 70/20/10 stratified split, SMOTE
3. **5 models trained**: LightGBM, CatBoost, XGBoost, Random Forest, Stacking
4. **Evaluated**: Validation set for selection, test set for final metrics
5. **Risk scores**: Low/Medium/High categories
6. **SHAP explanations**: Global and local interpretability
7. **CKD staging**: eGFR-based (Stages 1-5)
8. **Reports and visualizations** saved to outputs/

**Key design decisions:**
- eGFR, serum creatinine, ckd_stage excluded from prediction (data leakage)
- 70% Train / 20% Validation / 10% Test stratified split
- SMOTE (0.6 ratio) on training data only